In [5]:
# ======================================================
# Section 0.49 — Emergency Splits Backfill (aligned with 01–02)
#   • Writes out/splits/{X_train,X_val,X_test,y_train,y_val,y_test}.parquet
#   • Target resolution order: label (binary int) → label_str (map) → attack_cat (benign=0, else 1)
#   • Does NOT define canonicals; sequential-friendly.
# ======================================================
print(">>> Section 0.49 — Emergency Splits Backfill (01–02 aligned): start")

from pathlib import Path
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Respect existing globals; don't redefine canonicals
TARGET_COL = globals().get("TARGET_COL", "label")
SPLITS_DIR = Path(globals().get("SPLITS_DIR", "out/splits"))
DEFAULT_DATA = "archive/Payload_data_UNSW.csv"

LABEL_STR_MAP = {
    # negatives
    "benign": 0, "normal": 0, "noattack": 0, "false": 0, "neg": 0,
    # positives
    "malicious": 1, "attack": 1, "true": 1, "pos": 1,
}

def _resolve_data_path():
    gp = globals().get("DATA_PATH")
    if gp and os.path.exists(gp): return gp
    if os.path.exists(DEFAULT_DATA):
        if "DATA_PATH" not in globals():
            globals()["DATA_PATH"] = DEFAULT_DATA
        return DEFAULT_DATA
    raise FileNotFoundError(
        "[0.49] DATA_PATH not set and default dataset missing.\n"
        f"  • Expected: {DEFAULT_DATA}\n"
        "  • Or set DATA_PATH = '/path/to/UNSW.csv' in 01–02 Section 0.1."
    )

def _resolve_target(df: pd.DataFrame) -> pd.Series:
    """Mirror 01–02 target audit: label → label_str map → attack_cat."""
    # Case A: label exists and is numeric {0,1}
    if "label" in df.columns:
        try:
            y_num = pd.to_numeric(df["label"], errors="raise")
            uniq = set(pd.unique(y_num))
            if uniq.issubset({0,1}):
                return y_num.astype(int)
        except Exception:
            pass  # fall through

    # Case B: label_str with approved mapping
    if "label_str" in df.columns:
        s = df["label_str"].astype(str).str.strip().str.lower()
        mapped = s.map(LABEL_STR_MAP)
        unknown = sorted(set(mapped[s.notna()].index[s.isna()]) if False else list(set(s[mapped.isna()])))
        if len(unknown) == 0:
            return mapped.astype(int)
        # Unknown strings present; try attack_cat before failing
        print(f"[0.49][info] 'label_str' contains unmapped values (showing up to 10): {unknown[:10]}")

    # Case C: attack_cat (benign→0, everything else→1)
    if "attack_cat" in df.columns:
        ac = df["attack_cat"].astype(str).str.strip().str.lower()
        return (ac != "benign").astype(int)

    # If we get here, we cannot resolve a binary target—fail with audit
    cols = list(df.columns)[:20]
    raise AssertionError(
        "[0.49] Could not resolve a binary target using {label, label_str, attack_cat}.\n"
        f"Columns head: {cols}\n"
        "Fix in 01–02 (Section 0.2/0.3): ensure one of these is present and valid.\n"
        "DATA_SCHEME specifies target alternatives and mappings."
    )

def _exclude_feature_cols(df: pd.DataFrame) -> list:
    # From DATA_SCHEME.exclude_from_features: [TARGET_COL, 'payload', 'attack_cat', 'label_str']
    ex = set(["payload", "attack_cat", "label_str"])
    # Include the effective target name we used (prefer 'label' standard)
    if "label" in df.columns: ex.add("label")
    elif "label_str" in df.columns: ex.add("label_str")
    elif "attack_cat" in df.columns: ex.add("attack_cat")
    return [c for c in df.columns if c not in ex]

# Skip if splits already exist
req = ["X_train.parquet","X_val.parquet","X_test.parquet",
       "y_train.parquet","y_val.parquet","y_test.parquet"]
if all((SPLITS_DIR / f).exists() for f in req):
    print(f"[0.49] Splits already present at {SPLITS_DIR}; nothing to do.")
else:
    data_path = _resolve_data_path()
    print(f"[0.49] Using dataset at: {data_path}")
    df = pd.read_csv(data_path)

    # Resolve binary target using 01–02 rules
    y = _resolve_target(df)
    X = df[_exclude_feature_cols(df)]

    # Heads-up: if later sections require 'payload' column, ensure Section 2.1 runs beforehand
    if "payload" not in X.columns:
        print("[0.49][warn] 'payload' not in features. That’s fine for splits, "
              "but Section 2.1 must run before any payload vectorisation.")

    # 70/15/15 with stratify (RANDOM_STATE from 0.1 if available)
    RANDOM_STATE = globals().get("RANDOM_STATE", 42)
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
    )

    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    X_train.to_parquet(SPLITS_DIR / "X_train.parquet", index=False)
    X_val.to_parquet(SPLITS_DIR / "X_val.parquet", index=False)
    X_test.to_parquet(SPLITS_DIR / "X_test.parquet", index=False)
    # Persist as 'label' (standardised target name for downstream)
    y_train.to_frame("label").to_parquet(SPLITS_DIR / "y_train.parquet", index=False)
    y_val.to_frame("label").to_parquet(SPLITS_DIR / "y_val.parquet", index=False)
    y_test.to_frame("label").to_parquet(SPLITS_DIR / "y_test.parquet", index=False)

    print(f"[0.49] Wrote splits → {SPLITS_DIR} "
          f"(train={len(X_train)}, val={len(X_val)}, test={len(X_test)})")

# Reflect into memory so 05 can continue without reload
g = globals()
g["X_train"] = pd.read_parquet(SPLITS_DIR / "X_train.parquet")
g["X_val"]   = pd.read_parquet(SPLITS_DIR / "X_val.parquet")
g["X_test"]  = pd.read_parquet(SPLITS_DIR / "X_test.parquet")
g["y_train"] = pd.read_parquet(SPLITS_DIR / "y_train.parquet")["label"]
g["y_val"]   = pd.read_parquet(SPLITS_DIR / "y_val.parquet")["label"]
g["y_test"]  = pd.read_parquet(SPLITS_DIR / "y_test.parquet")["label"]
print("[0.49] Splits available in memory.")

print(">>> Section 0.49 — Emergency Splits Backfill (01–02 aligned): complete")


>>> Section 0.49 — Emergency Splits Backfill (01–02 aligned): start
[0.49] Splits already present at out/splits; nothing to do.
[0.49] Splits available in memory.
>>> Section 0.49 — Emergency Splits Backfill (01–02 aligned): complete


## Section 0.5 — Preflight Bootstrap

Ensures splits exist (or rebuilds from `DATA_PATH`) and keeps HPO cells fail-fast but reproducible.

In [6]:
# >>> Section 0.5 — Preflight Bootstrap (robust splits for 05)
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp  # used elsewhere in 05
from sklearn.model_selection import train_test_split

# Canonical config
TARGET_COL   = globals().get("TARGET_COL", "label")
SPLITS_DIR   = Path(globals().get("SPLITS_DIR", "out/splits"))
DEFAULT_DATA = "archive/Payload_data_UNSW.csv"
VEC_PATH     = Path(globals().get("VEC_PATH", "out/payload/vectorizer.joblib"))

def _resolve_data_path():
    """Order: notebook global → ENV → default file. Fail-fast with clear instructions."""
    gp = globals().get("DATA_PATH", None)
    if gp and os.path.exists(gp):
        return gp
    ep = os.environ.get("DATA_PATH", None)
    if ep and os.path.exists(ep):
        globals()["DATA_PATH"] = ep
        return ep
    if os.path.exists(DEFAULT_DATA):
        globals()["DATA_PATH"] = DEFAULT_DATA
        return DEFAULT_DATA
    raise FileNotFoundError(
        "[0.5] Pre-split dataset not found.\n"
        f"  Tried: globals().get('DATA_PATH')={repr(gp)}, ENV DATA_PATH={repr(ep)}, DEFAULT={DEFAULT_DATA}\n"
        "Fix one:\n"
        "  • Put your CSV at 'archive/Payload_data_UNSW.csv', or\n"
        "  • Set in-notebook: DATA_PATH='path/to/your.csv', or\n"
        "  • Shell: export DATA_PATH='path/to/your.csv'\n"
        "Then re-run Section 0.5."
    )

def _persist_splits(X_train, X_val, X_test, y_train, y_val, y_test, out_dir=SPLITS_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    X_train.to_parquet(out_dir / "X_train.parquet", index=False)
    X_val.to_parquet(out_dir / "X_val.parquet", index=False)
    X_test.to_parquet(out_dir / "X_test.parquet", index=False)
    y_train.to_frame(TARGET_COL).to_parquet(out_dir / "y_train.parquet", index=False)
    y_val.to_frame(TARGET_COL).to_parquet(out_dir / "y_val.parquet", index=False)
    y_test.to_frame(TARGET_COL).to_parquet(out_dir / "y_test.parquet", index=False)

def _load_splits(in_dir=SPLITS_DIR):
    X_train = pd.read_parquet(in_dir / "X_train.parquet")
    X_val   = pd.read_parquet(in_dir / "X_val.parquet")
    X_test  = pd.read_parquet(in_dir / "X_test.parquet")
    y_train = pd.read_parquet(in_dir / "y_train.parquet")[TARGET_COL]
    y_val   = pd.read_parquet(in_dir / "y_val.parquet")[TARGET_COL]
    y_test  = pd.read_parquet(in_dir / "y_test.parquet")[TARGET_COL]
    return X_train, X_val, X_test, y_train, y_val, y_test

def _splits_exist(in_dir=SPLITS_DIR):
    req = ["X_train.parquet","X_val.parquet","X_test.parquet",
           "y_train.parquet","y_val.parquet","y_test.parquet"]
    return all((in_dir / f).exists() for f in req)

def _ensure_dataframe_splits(test_size=0.15, val_size=0.15, random_state=42):
    """
    Drop-in replacement for 05:
    - If splits exist: load + expose to globals.
    - Else: resolve DATA_PATH (global/env/default), rebuild with stratification, persist, then expose.
    """
    print(">>> Section 0.5 — Splits Bootstrapper")
    if _splits_exist():
        print("[0.5] Found existing Parquet splits under", SPLITS_DIR)
        X_train, X_val, X_test, y_train, y_val, y_test = _load_splits()
    else:
        data_path = _resolve_data_path()
        print(f"[0.5] Rebuilding splits from DATA_PATH={data_path}")
        df = pd.read_csv(data_path)

        # Validate & coerce target
        assert TARGET_COL in df.columns, f"[0.5] TARGET_COL '{TARGET_COL}' missing. Columns head: {list(df.columns)[:10]}"
        y = df[TARGET_COL]
        if y.dtype.kind not in "iu":
            mapping = {True:1, False:0, "malicious":1, "benign":0, "attack":1, "normal":0}
            y = y.map(lambda v: mapping.get(v, v))
            y = pd.to_numeric(y, errors="raise")
        assert set(pd.unique(y)).issubset({0,1}), "[0.5] TARGET_COL must be binary in {0,1}."

        X = df.drop(columns=[TARGET_COL])

        if 'payload' not in X.columns:
            print("[0.5][warn] 'payload' not in features. Re-run your payload folding (2.x) if required.")

        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=(val_size + test_size), random_state=random_state, stratify=y
        )
        rel_test = test_size / (val_size + test_size)
        X_val, X_test, y_val, y_test = train_test_split(
            X_tmp, y_tmp, test_size=rel_test, random_state=random_state, stratify=y_tmp
        )

        _persist_splits(X_train, X_val, X_test, y_train, y_val, y_test)
        print(f"[0.5] Wrote splits → {SPLITS_DIR} (train={len(X_train)}, val={len(X_val)}, test={len(X_test)})")

    g = globals()
    g["X_train"], g["X_val"], g["X_test"] = X_train, X_val, X_test
    g["y_train"], g["y_val"], g["y_test"] = y_train, y_val, y_test
    print("[0.5] Splits available in memory: X_/y_ for train/val/test")


## Section 5.0 — Payload Bootstrap (rebuild features in this kernel)

In [7]:
# ======================================================
# Section 5.0 — Preflight for HPO (sequential pipeline)
#   • Load persisted splits from out/splits/
#   • Verify & load vectorizer built in 2.1
# ======================================================
print(">>> Section 5.0 — Preflight: start")

from pathlib import Path
import pandas as pd
import joblib

# Respect existing globals without redefining canonical config
TARGET_COL = globals().get("TARGET_COL", "label")
SPLITS_DIR = Path(globals().get("SPLITS_DIR", "out/splits"))
VEC_PATH   = Path(globals().get("VEC_PATH", "out/payload/vectorizer.joblib"))

def _require_splits_for_05():
    req = ["X_train.parquet","X_val.parquet","X_test.parquet",
           "y_train.parquet","y_val.parquet","y_test.parquet"]
    missing = [f for f in req if not (SPLITS_DIR / f).exists()]
    if missing:
        raise RuntimeError(
            "\n".join([
                "[5.0] Splits not found for sequential run.",
                f"Missing: {missing}",
                "Remediation:",
                "  • Run 01–02 up to Section 0.4 (Split & Preprocessing) to persist splits,",
                "  • Then run Section 2.1 (Payload Sequence / Vectorizer) to create VEC_PATH.",
                "  • In Section 0.1 of 01, ensure DATA_PATH is set (default: archive/Payload_data_UNSW.csv).",
            ])
        )
    X_train = pd.read_parquet(SPLITS_DIR / "X_train.parquet")
    X_val   = pd.read_parquet(SPLITS_DIR / "X_val.parquet")
    X_test  = pd.read_parquet(SPLITS_DIR / "X_test.parquet")
    y_train = pd.read_parquet(SPLITS_DIR / "y_train.parquet")[TARGET_COL]
    y_val   = pd.read_parquet(SPLITS_DIR / "y_val.parquet")[TARGET_COL]
    y_test  = pd.read_parquet(SPLITS_DIR / "y_test.parquet")[TARGET_COL]
    g = globals()
    g["X_train"], g["X_val"], g["X_test"] = X_train, X_val, X_test
    g["y_train"], g["y_val"], g["y_test"] = y_train, y_val, y_test
    print(f"[5.0] Loaded splits from {SPLITS_DIR} "
          f"(train={len(X_train)}, val={len(X_val)}, test={len(X_test)})")

# Load splits (fail-fast with pointer back to 01–02 if absent)
_require_splits_for_05()

# Vectorizer must already exist from 2.1
if not VEC_PATH.exists():
    raise AssertionError(
        f"[5.0] Vectorizer missing at {VEC_PATH}. "
        "Run 2.1 — Payload Sequence / Vectorizer in 01–02 before 05."
    )
vectorizer = joblib.load(VEC_PATH)
print(f"[5.0] Loaded vectorizer from {VEC_PATH}")

print(">>> Section 5.0 — Preflight: complete")


>>> Section 5.0 — Preflight: start
[5.0] Loaded splits from out/splits (train=15545, val=3331, test=3332)
[5.0] Loaded vectorizer from out/payload/vectorizer.joblib
>>> Section 5.0 — Preflight: complete


In [8]:
# ======================================================
# Section 5.0b — Build Payload Features for HPO
#   • Fold payload_byte_* → 'payload' strings (in-memory only)
#   • Transform with fitted TF-IDF vectorizer from 2.1
#   • Expose: X_train_payload, X_val_payload (csr matrices)
# ======================================================
print(">>> Section 5.0b — Payload features: start")

import re
import numpy as np
import scipy.sparse as sp

# Preconditions from 5.0
for nm in ["X_train", "X_val", "y_train", "y_val", "vectorizer"]:
    assert nm in globals(), f"[5.0b] Missing {nm}. Run 5.0 (and 2.1 in 01–02) first."

# 1) Detect payload bytes in splits
def _payload_byte_cols(df):
    return [c for c in df.columns if re.match(r"^payload_byte_\d+$", c)]

byte_cols_tr = _payload_byte_cols(X_train)
byte_cols_va = _payload_byte_cols(X_val)
assert byte_cols_tr and byte_cols_va, (
    "[5.0b] No payload_byte_* columns found in X_train/X_val. "
    "Ensure 01–02 Section 0.4 did NOT drop them, or persist 'payload' strings in splits."
)
assert byte_cols_tr == byte_cols_va, (
    "[5.0b] Mismatch in payload byte columns between train/val."
)

# 2) Fold bytes → payload string (space-separated integers)
def _fold_payload_bytes(df, cols):
    # Make sure we don't stringify NaNs to 'nan'
    arr = df[cols].astype("Int64").astype(str).replace("<NA>", "0")
    return arr.agg(" ".join, axis=1)

payload_train = _fold_payload_bytes(X_train, byte_cols_tr)
payload_val   = _fold_payload_bytes(X_val, byte_cols_va)

# 3) Transform with fitted vectorizer
Xs_tr = vectorizer.transform(payload_train.astype(str))
Xs_va = vectorizer.transform(payload_val.astype(str))

# 4) Expose for 5.1+
globals()["X_train_payload"] = Xs_tr.tocsr() if not sp.isspmatrix_csr(Xs_tr) else Xs_tr
globals()["X_val_payload"]   = Xs_va.tocsr() if not sp.isspmatrix_csr(Xs_va) else Xs_va

# y already in memory from 5.0; ensure dtype
globals()["y_train"] = np.asarray(y_train).astype(int)
globals()["y_val"]   = np.asarray(y_val).astype(int)

print(f"[5.0b] X_train_payload shape: {globals()['X_train_payload'].shape}")
print(f"[5.0b] X_val_payload   shape: {globals()['X_val_payload'].shape}")
print(">>> Section 5.0b — Payload features: complete")


>>> Section 5.0b — Payload features: start
[5.0b] X_train_payload shape: (15545, 256)
[5.0b] X_val_payload   shape: (3331, 256)
>>> Section 5.0b — Payload features: complete


## Section 5.1 — HPO with Bayesian Optimization (payload-only)

In [9]:

print(">>> Section 5.1: HPO with Bayesian Optimization [TIME-BOXED]")
import time, json, joblib, numpy as np
from pathlib import Path
from sklearn.metrics import average_precision_score
import lightgbm as lgb
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args
from skopt.callbacks import CheckpointSaver

for nm in ["X_train_payload","y_train","X_val_payload","y_val"]:
    assert nm in globals(), f"[5.1] Missing {nm}. Run 5.0 first."

TIME_BUDGET_SEC    = int(globals().get("BO_TIME_BUDGET_SEC", 420))
N_CALLS_MAX        = int(globals().get("BO_N_CALLS_MAX", 20))
N_RANDOM_STARTS    = int(globals().get("BO_N_RANDOM_STARTS", 5))
EARLY_STOP_ROUNDS  = int(globals().get("BO_EARLY_STOP", 20))
NUM_BOOST_ROUND    = int(globals().get("BO_NUM_BOOST_ROUND", 500))
SUBSAMPLE_FRACTION = float(globals().get("BO_SUBSAMPLE_FRAC", 1.0))
RNG_SEED           = int(globals().get("RANDOM_STATE", 1337))
N_THREADS          = int(globals().get("N_THREADS", -1))

BO_STATE = Path(OUT_ROOT if 'OUT_ROOT' in globals() else "out") / "bo_state.pkl"
BO_STATE.parent.mkdir(parents=True, exist_ok=True)

def _maybe_subsample(X, y, frac, seed):
    if frac >= 0.999: 
        return X, y
    n = X.shape[0]
    k = max(1000, int(n * frac))
    y_arr = np.asarray(y)
    pos_idx = np.where(y_arr == 1)[0]
    neg_idx = np.where(y_arr == 0)[0]
    k_pos = max(1, int(k * len(pos_idx) / max(1, len(y_arr))))
    k_neg = max(1, k - k_pos)
    rng = np.random.RandomState(seed)
    take = np.r_[rng.choice(pos_idx, size=min(k_pos, len(pos_idx)), replace=False),
                 rng.choice(neg_idx, size=min(k_neg, len(neg_idx)), replace=False)]
    rng.shuffle(take)
    return X[take], y_arr[take]

Xtr_bo, ytr_bo = _maybe_subsample(X_train_payload, y_train, SUBSAMPLE_FRACTION, RNG_SEED)

space = [
    Real(0.01, 0.3,   name="learning_rate", prior="log-uniform"),
    Integer(16, 256,  name="num_leaves"),
    Integer(2,  12,   name="max_depth"),
    Integer(5,  200,  name="min_child_samples"),
    Real(0.4,  1.0,   name="feature_fraction"),
    Real(0.4,  1.0,   name="bagging_fraction"),
    Real(0.0,  10.0,  name="lambda_l1"),
    Real(0.0,  10.0,  name="lambda_l2"),
]

@use_named_args(space)
def objective(**p):
    params = dict(
        objective="binary",
        metric="binary_logloss",
        learning_rate=float(p["learning_rate"]),
        num_leaves=int(p["num_leaves"]),
        max_depth=int(p["max_depth"]),
        min_child_samples=int(p["min_child_samples"]),
        feature_fraction=float(p["feature_fraction"]),
        bagging_fraction=float(p["bagging_fraction"]),
        lambda_l1=float(p["lambda_l1"]),
        lambda_l2=float(p["lambda_l2"]),
        feature_pre_filter=False,
        verbosity=-1,
        seed=RNG_SEED,
        n_jobs=N_THREADS,
    )
    dtrain = lgb.Dataset(Xtr_bo, label=ytr_bo, free_raw_data=True)
    dval   = lgb.Dataset(X_val_payload, label=y_val, reference=dtrain, free_raw_data=True)
    booster = lgb.train(
        params,
        dtrain,
        valid_sets=[dval],
        num_boost_round=NUM_BOOST_ROUND,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP_ROUNDS, verbose=False)],
    )
    y_pred = booster.predict(X_val_payload, num_iteration=booster.best_iteration)
    ap = average_precision_score(y_val, y_pred)
    print(f"[5.1] Trial AP={ap:.5f}  (iters={booster.best_iteration})", flush=True)
    return -float(ap)  # skopt minimizes

start = time.time()
def _time_guard(res):
    if (time.time() - start) > TIME_BUDGET_SEC:
        print(f"[5.1] Time budget hit ({TIME_BUDGET_SEC}s). Stopping BO.", flush=True)
        return True
    return False

callbacks = [CheckpointSaver(str(BO_STATE), compress=3), _time_guard]

print(f"[5.1] Starting BO: n_calls≤{N_CALLS_MAX}, random_starts={N_RANDOM_STARTS}, "
      f"time_budget={TIME_BUDGET_SEC}s, subsample_frac={SUBSAMPLE_FRACTION}", flush=True)

from skopt import gp_minimize
res = gp_minimize(
    func=objective,
    dimensions=space,
    n_calls=N_CALLS_MAX,
    n_initial_points=N_RANDOM_STARTS,
    acq_func="EI",
    random_state=RNG_SEED,
    callback=callbacks,
    n_jobs=1,
)

best_ap = -float(res.fun)
best_params = {dim.name: val for dim, val in zip(space, res.x)}
print(f"[5.1] Best Val AP={best_ap:.5f}")
print("[5.1] Best params:", best_params)

# NumPy-safe JSON summary
def _to_py(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)):    return bool(obj)
    if isinstance(obj, (np.ndarray,)):  return obj.tolist()
    return obj
def _map_py(o):
    if isinstance(o, dict):  return {str(k): _map_py(_to_py(v)) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_map_py(_to_py(v)) for v in o]
    return _to_py(o)

out_root = OUT_ROOT if 'OUT_ROOT' in globals() else Path("out")
Path(out_root).mkdir(parents=True, exist_ok=True)
(Path(out_root) / "bo_summary.json").write_text(json.dumps(_map_py({"best_val_ap": best_ap, "best_params": best_params, "calls": len(res.x_iters)}), indent=2))
print(f"[5.1] Summary → {(Path(out_root)/'bo_summary.json')}")


>>> Section 5.1: HPO with Bayesian Optimization [TIME-BOXED]
[5.1] Starting BO: n_calls≤20, random_starts=5, time_budget=420s, subsample_frac=1.0
[5.1] Trial AP=0.99935  (iters=200)
[5.1] Trial AP=0.99462  (iters=495)
[5.1] Trial AP=0.99641  (iters=500)
[5.1] Trial AP=0.99621  (iters=500)
[5.1] Trial AP=0.99809  (iters=357)
[5.1] Trial AP=0.99997  (iters=339)
[5.1] Trial AP=1.00000  (iters=239)
[5.1] Trial AP=0.99997  (iters=288)
[5.1] Trial AP=0.99910  (iters=500)
[5.1] Trial AP=0.99988  (iters=120)
[5.1] Trial AP=1.00000  (iters=145)
[5.1] Trial AP=0.98363  (iters=500)
[5.1] Trial AP=0.99714  (iters=30)
[5.1] Trial AP=0.99472  (iters=185)
[5.1] Trial AP=0.99960  (iters=295)
[5.1] Trial AP=0.99888  (iters=500)
[5.1] Trial AP=1.00000  (iters=500)
[5.1] Trial AP=0.99814  (iters=500)
[5.1] Trial AP=0.99845  (iters=500)
[5.1] Trial AP=0.99997  (iters=237)
[5.1] Best Val AP=1.00000
[5.1] Best params: {'learning_rate': 0.06668410419748122, 'num_leaves': 48, 'max_depth': 6, 'min_child_sample

## Section 5.2 — Train Final Model with Best Params & Persist Bundle

In [10]:

print(">>> Section 5.2: Train Final Model with Best Params & Persist Bundle")
import json, shutil, numpy as np, lightgbm as lgb, joblib
from sklearn.metrics import average_precision_score
from pathlib import Path

assert "best_params" in globals(), "[5.2] No best_params from 5.1"

params = dict(
    objective="binary",
    metric="binary_logloss",
    feature_pre_filter=False,
    verbosity=-1,
    seed=int(globals().get("RANDOM_STATE", 1337)),
    n_jobs=int(globals().get("N_THREADS", -1)),
    **{k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v)
       for k,v in best_params.items()}
)

dtrain = lgb.Dataset(X_train_payload, label=y_train)
dval   = lgb.Dataset(X_val_payload, label=y_val, reference=dtrain)

model = lgb.train(
    params,
    dtrain,
    valid_sets=[dtrain, dval],
    num_boost_round=int(globals().get("FINAL_NUM_BOOST_ROUND", 800)),
    callbacks=[lgb.early_stopping(stopping_rounds=int(globals().get("FINAL_EARLY_STOP", 50)), verbose=False)]
)

# Metrics
y_tr_pred = model.predict(X_train_payload, num_iteration=model.best_iteration)
y_va_pred = model.predict(X_val_payload,   num_iteration=model.best_iteration)
train_ap = float(average_precision_score(y_train, y_tr_pred))
val_ap   = float(average_precision_score(y_val,   y_va_pred))
test_ap  = None
if "X_test_payload" in globals() and "y_test" in globals():
    y_te_pred = model.predict(X_test_payload, num_iteration=model.best_iteration)
    test_ap = float(average_precision_score(y_test, y_te_pred))

print(f"[5.2] Train AP={train_ap:.5f}; Val AP={val_ap:.5f}; Test AP={(None if test_ap is None else round(test_ap,5))}")

# Persist model + vectorizer + manifest
out_root = Path(OUT_ROOT) if 'OUT_ROOT' in globals() else Path("out")
out_root.mkdir(parents=True, exist_ok=True)
model_path = out_root / "hpo_lgbm_payload.pkl"
joblib.dump(model, model_path)

sec2_dir = (Path(STAGE_ROOT) if 'STAGE_ROOT' in globals() else Path("staging")) / "section_2"
vec_src = sec2_dir / "vectorizer.joblib"
vec_dst = out_root / "vectorizer.joblib"
if vec_src.exists():
    shutil.copy2(vec_src, vec_dst)
else:
    vec_dst = None

def _to_py(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)):    return bool(obj)
    if isinstance(obj, (np.ndarray,)):  return obj.tolist()
    return obj
def _map_py(o):
    if isinstance(o, dict):  return {str(k): _map_py(_to_py(v)) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_map_py(_to_py(v)) for v in o]
    return _to_py(o)

bundle = {
    "model_path": str(model_path.resolve()),
    "vectorizer_path": (str(vec_dst.resolve()) if vec_dst else None),
    "best_params": _map_py(best_params),
    "metrics": {"train_ap": train_ap, "val_ap": val_ap, "test_ap": test_ap},
    "random_state": int(globals().get("RANDOM_STATE", 1337)),
}
(out_root / "HPO_MODEL_BUNDLE.json").write_text(json.dumps(_map_py(bundle), indent=2))
print(f"[5.2] Bundle → {out_root/'HPO_MODEL_BUNDLE.json'}")


>>> Section 5.2: Train Final Model with Best Params & Persist Bundle
[5.2] Train AP=1.00000; Val AP=1.00000; Test AP=None
[5.2] Bundle → out/HPO_MODEL_BUNDLE.json
